In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st
import numpy as np

In [ ]:
homer_path = ''
env = ''

In [ ]:
os.system(env+'python ./metilene3/metilene3.py \
    -i ./data/GSE222147_pdac.input.tsv \
    -o ./PDAC \
    -t 16 \
    -n 4 \
    -plot True -anno hg19\
')

In [ ]:
os.system('cp ./PDAC/DMRs-unsupervised.tsv ../SourceData/Fig.6b.txt')

In [ ]:
renameG = {
    'G0':'A',
    'G1':'B',
    'G2':'C',
    'G3':'D',
    'G4':'E',
    'G5':'F',
}

pdac_s = pd.read_table('./PDAC/DMRs.tsv')
pdac_s['hypomethylated'] = pdac_s['Hypo-groups']
pdac_s['intermediate'] = pdac_s['Int-groups']
pdac_s['hypermethylated'] = pdac_s['Hyper-groups']
for i in renameG.keys():
    pdac_s['hypomethylated'] = pdac_s['hypomethylated'].str.replace(i,renameG[i])
    pdac_s['intermediate'] = pdac_s['intermediate'].str.replace(i,renameG[i])
    pdac_s['hypermethylated'] = pdac_s['hypermethylated'].str.replace(i,renameG[i])
pdac_s['mode'] = 'supervised'

# pdac = pd.concat([pdac_u, pdac_s]).sort_values(['mode','chr','start'])
pdac = pd.concat([pdac_s]).sort_values(['length','p-kwt'], ascending=[0,1])
pdac = pdac['chr	start	stop	meandiffabs	length	p-kwt	hypomethylated	intermediate	hypermethylated'.split('\t')]
pdac.to_csv('./figures/ST7.tsv',sep='\t', index=False)
pdac

In [ ]:
dmrmean_m_rename = pd.read_table('./PDAC/heatmap.tsv', index_col=0)
dmrmean_m_rename
colors = dmrmean_m_rename[[]]

colors['group'] = [i.split(' ')[0] for i in dmrmean_m_rename.index]
colors['group'] = colors['group'].map({
    'G0':sns.color_palette("Paired")[4],
    'G1':sns.color_palette("Paired")[8],
    'G2':sns.color_palette("Paired")[2],
    'G3':sns.color_palette("Paired")[0],
})
colors['subtype'] = [i.split('=')[1].split('_')[0] for i in dmrmean_m_rename.index]

typec = {
        'PDAC':sns.color_palette("Paired")[5],
        'PanIN':sns.color_palette("Paired")[9],
        'Acinar':sns.color_palette("Paired")[3],
        'Duct':sns.color_palette("Paired")[1],
    }

In [ ]:
colors = colors.loc[dmrmean_m_rename.index]
cm = sns.clustermap(dmrmean_m_rename,\
        row_colors=[colors['group'],\
                    (colors['subtype']==1).map({False:'white'}),\
                    colors['subtype'].map(typec),\
                    (colors['subtype']==1).map({False:'white'}),\
                    ],\
        # row_linkage=lk.linkage,\
        col_cluster=False,row_cluster=False,\
        cmap='Spectral_r', dendrogram_ratio=0.000001, xticklabels=False, yticklabels=False, \
        method='ward', cbar_pos=None, vmax=1, vmin=0, center=0.5, colors_ratio=0.03, figsize=[6,10])
plt.savefig('./figures/6b-r.pdf', bbox_inches='tight')

In [ ]:
from Bio import Phylo

tree = Phylo.read("./PDAC/DMTree.nwk", "newick")

def change_labels(clade):
    if clade.name:
        clade.name = clade.name.split('=')[-1]+'-'.join(['' for i in range(20)])
    for subclade in clade.clades:
        change_labels(subclade)

change_labels(tree.root)

cmap = colors['group'].to_dict()
for i in colors.index:
    cmap[i.split('=')[-1]+'-'.join(['' for i in range(20)])] = cmap[i]
f,a = plt.subplots(figsize=[5,8])
Phylo.draw(tree, axes=a, do_show=False, label_colors=cmap,show_confidence=True)
plt.xscale('symlog')
plt.xlim([-0.1,1e5/2])
a.spines['top'].set_visible(False)
a.spines['left'].set_visible(False)
a.spines['right'].set_visible(False)
a.yaxis.set_visible(False)
plt.savefig('./figures/6b.pdf', bbox_inches='tight')

In [ ]:
sinfo = dmrmean_m_rename[[]]
sinfo['group'] = [i.split(' ')[0] for i in dmrmean_m_rename.index]
sinfo['subtype'] = [i.split('=')[1].split('_')[0] for i in dmrmean_m_rename.index]
sinfo.index = [i.split('=')[-1] for i in sinfo.index]

In [ ]:
met = pd.read_table('./data/GSE222147_pdac.input.tsv', na_values=['.']).dropna()
met.columns = [i.split('=')[-1] for i in met.columns]
met

In [ ]:
cpgstd = met.drop(columns=['chr','pos']).T.std()
cpgstd

In [ ]:
udmrs = pd.read_table('./PDAC/DMRs-unsupervised.tsv',skiprows=2)
dmtncpg = udmrs.loc[(udmrs['meandiffabs']>0.5)&(udmrs['#Hyper']>=2)&(udmrs['#Hypo']>=2)]['length'].sum()
dmtncpg

In [ ]:
udmr4pca = []
udmrs.loc[(udmrs['meandiffabs']>0.5)&(udmrs['#Hyper']>=2)&(udmrs['#Hypo']>=2)]['mean'].apply(lambda x:udmr4pca.append(x.split('|')))
udmr4pca = pd.DataFrame(udmr4pca).astype(float).T
udmr4pca.index = met.columns[2:]
udmr4pca

In [ ]:
f,ax = plt.subplots(1,4,figsize=[12,3])

clsc = {
    'G0':sns.color_palette("Paired")[5],
    'G1':sns.color_palette("Paired")[9],
    'G2':sns.color_palette("Paired")[3],
    'G3':sns.color_palette("Paired")[1],
}

import numpy as np
from sklearn.decomposition import PCA

sd_pdac_pcas = []
models = ['all CpGs','top 1% CpGs','eq. #CpGs','unsupervised DMRs']

for ii, tmp in enumerate([
    np.array(met.drop(columns='chr	pos'.split('\t')).T),
    np.array(met.drop(columns='chr	pos'.split('\t')).loc[cpgstd>cpgstd.quantile(0.99)].T),
    np.array(met.drop(columns='chr	pos'.split('\t')).loc[cpgstd>=list(cpgstd.sort_values())[-dmtncpg]].T),
    udmr4pca
]):
    print(tmp.shape)
    pca = PCA(n_components=2)
    X = pd.DataFrame(pca.fit_transform(tmp))
    X['model'] = models[ii]
    sd_pdac_pcas.append(X)
    print(pca.explained_variance_ratio_)
    X.index = met.columns[2:]
    X['grp'] = X.index.map(sinfo['group'])
    X['subtype'] = X.index.map(sinfo['subtype'])
    X['batch'] = X.index.str.contains('_B')
    a = sns.scatterplot(x=X[0],y=X[1],hue=X['grp'],s=50, style=X['batch'],palette=clsc, ax=ax[ii], legend=None)
    a.spines['top'].set_visible(False)
    # a.spines['left'].set_visible(False)
    a.spines['right'].set_visible(False)
    # a.yaxis.set_visible(False)
    a.set_title('n='+str(tmp.shape[1]))
    a.set_ylabel('PC2('+str(pca.explained_variance_ratio_[1]*100)[:5]+'%)')
    a.set_xlabel('PC1('+str(pca.explained_variance_ratio_[0]*100)[:5]+'%)')
    a.set_box_aspect(1)
    a.set_xticks(np.linspace(*a.get_xlim(), 3))
    a.set_yticks(np.linspace(*a.get_ylim(), 3))

f.tight_layout()
plt.savefig('./figures/6c.pdf', bbox_inches='tight')

In [ ]:
pd.concat(sd_pdac_pcas)[[0,1,'model']].to_csv('../SourceData/Fig.6c.txt',sep='\t')
pd.concat(sd_pdac_pcas)

In [ ]:
dmrs = pd.read_table('./PDAC/DMRs.tsv').sort_values(['chr','start','stop'])
dmrs['strand'] = '+'
dmrs['stop_1'] = dmrs['stop']+1

In [ ]:
os.system('cp ./PDAC/DMRs.tsv ../SourceData/Fig.6de.txt')

In [ ]:
linedist = 0.5
f,a = plt.subplots(figsize=[10,2])

import numpy as np

for cutoff in [0.1,0.3,0.5]:
    ctdmrs = dmrs.loc[dmrs['sig.comparison'].apply(lambda x:x[-3:] in ['1|3','3|1']) & \
                    (dmrs['meandiffabs']>cutoff)]
    
    ctdmrs_PDAC = ctdmrs.loc[ctdmrs['sig.comparison'].apply(lambda x:x[0] in ['1','3'])]
    ctdmrs_panIN = ctdmrs.loc[ctdmrs['sig.comparison'].apply(lambda x:x[2] in ['1','3'])]
        

    x_PDAC = ctdmrs_PDAC['sig.comparison'].apply(lambda x:(x[0]==x[-1])) # PDAC==Duct
    x_panIN = ctdmrs_panIN['sig.comparison'].apply(lambda x:(x[2]==x[-1])) # PanIN==Duct
    print(x_panIN.sum(),(1-x_panIN).sum(),x_PDAC.sum(),(1-x_PDAC).sum())

    print(st.fisher_exact([
        [x_panIN.sum(),(1-x_panIN).sum()],
        [x_PDAC.sum(),(1-x_PDAC).sum()]
    ]))
    
    y = linedist*(cutoff-0.3)*-5
    
    a = sns.scatterplot(x=[0, x_panIN.mean(), x_PDAC.mean(), 1],y=[y for i in range(4)], s=100, linewidth=1, \
                        color=[sns.color_palette("Paired")[3], sns.color_palette("Paired")[9], \
                               sns.color_palette("Paired")[5], sns.color_palette("Paired")[1]], edgecolor=None, zorder=2)
    a = sns.lineplot(x=[0, 1],y=[y, y], linewidth=1, color='lightgrey', zorder=0)
    a = sns.lineplot(x=[x_panIN.mean(), x_PDAC.mean()],y=[y, y], linewidth=2, color='red', zorder=1)


a = sns.lineplot(x=[0, 1],y=[-1.25,-1.25], linewidth=1, color='black', zorder=0)
a = sns.lineplot(x=[1, 1],y=[-1.25,-0.9], linewidth=1, color='black', zorder=0, estimator=None)
a = sns.lineplot(x=[0, 1],y=[1.25,1.25], linewidth=1, color='black', zorder=0)
a = sns.lineplot(x=[0, 0],y=[1.25,0.9], linewidth=1, color='black', zorder=0, estimator=None)
a = sns.lineplot(x=[0.5, 0.5],y=[1.23,-1.23], linewidth=1, color='lightgrey', linestyle='--', zorder=0, estimator=None)


plt.ylim([-1.5,1.5])
plt.xticks([0,0.5,1])
plt.xlim([-0.03,1.03])
a.spines['top'].set_visible(False)
a.spines['bottom'].set_visible(False)
a.spines['left'].set_visible(False)
a.spines['right'].set_visible(False)
a.yaxis.set_visible(False)
a.xaxis.set_visible(False)
plt.savefig('./figures/6d.pdf', bbox_inches='tight')

In [ ]:
tmp = dmrs.loc[(dmrs['sig.comparison'].apply(lambda x:x[-3:] in ['1|3',])) & \
            (dmrs['meandiffabs']>0.5)]
tmp['pdac'] = tmp['mean'].apply(lambda x:float(x.split('|')[0]))
tmp['neo'] = tmp['mean'].apply(lambda x:float(x.split('|')[1]))
tmp['diff'] = (dmrs['sig.comparison'].apply(lambda x:x[-3:]))
tmp['diff2'] = (dmrs['sig.comparison'].apply(lambda x:x[:3]))

tmp2 = []
tmp.sort_values(['diff','diff2'])['mean'].apply(lambda x:tmp2.append(x.split('|')))

tmp2 = pd.melt(pd.DataFrame(tmp2).astype(float))
tmp2['variable'] = tmp2['variable'].map({0:'PDAC',1:'PanIN',2:'Acinar',3:'Duct'})

print(tmp2['variable'].value_counts())

for i in ['Acinar','PanIN','PDAC','Duct']:
    for j in ['Acinar','PanIN','PDAC','Duct']:
        if i>j:
            print(i,j,st.ranksums(tmp2.loc[tmp2['variable']==i]['value'],\
                              tmp2.loc[tmp2['variable']==j]['value']))

plt.subplots(figsize=[2,3])
a=sns.boxplot(x=tmp2['variable'], y=tmp2['value'], \
              order=['Acinar','PanIN','PDAC','Duct'],\
              # color=tmp2['variable'],\
              palette=typec, \
              color='white', width=.5)
a.spines['top'].set_visible(False)
a.spines['right'].set_visible(False)
a.set_yticks([0,0.5,1])
a.set_ylim([0,1])
a.set_ylabel(None)
a.set_xlabel(None)
a.tick_params(axis='x', labelrotation=60)
# a.yaxis.set_visible(False)
plt.savefig('./figures/6e.pdf', bbox_inches='tight')

In [ ]:
tmp = dmrs.loc[(dmrs['sig.comparison'].apply(lambda x:x[-3:] in ['3|1',])) & \
            (dmrs['meandiffabs']>0.5)]
tmp['pdac'] = tmp['mean'].apply(lambda x:float(x.split('|')[0]))
tmp['neo'] = tmp['mean'].apply(lambda x:float(x.split('|')[1]))
tmp['diff'] = (dmrs['sig.comparison'].apply(lambda x:x[-3:]))
tmp['diff2'] = (dmrs['sig.comparison'].apply(lambda x:x[:3]))

tmp2 = []
tmp.sort_values(['diff','diff2'])['mean'].apply(lambda x:tmp2.append(x.split('|')))

tmp2 = pd.melt(pd.DataFrame(tmp2).astype(float))
tmp2['variable'] = tmp2['variable'].map({0:'PDAC',1:'PanIN',2:'Acinar',3:'Duct'})

print(tmp2['variable'].value_counts())

for i in ['Acinar','PanIN','PDAC','Duct']:
    for j in ['Acinar','PanIN','PDAC','Duct']:
        if i>j:
            print(i,j,st.ranksums(tmp2.loc[tmp2['variable']==i]['value'],\
                              tmp2.loc[tmp2['variable']==j]['value']))

plt.subplots(figsize=[2,3])
a=sns.boxplot(x=tmp2['variable'], y=tmp2['value'], \
              order=['Acinar','PanIN','PDAC','Duct'],\
              # color=tmp2['variable'],\
              palette=typec, \
              color='white', width=.5)
a.spines['top'].set_visible(False)
a.spines['right'].set_visible(False)
a.set_yticks([0,0.5,1])
a.set_ylim([0,1])
a.set_ylabel(None)
a.set_xlabel(None)
a.tick_params(axis='x', labelrotation=60)
# a.yaxis.set_visible(False)
plt.savefig('./figures/6e-r.pdf', bbox_inches='tight')

In [ ]:
path = './PDAC/motif/'
os.system('mkdir '+path)
for i in ['P2|3|3|3']:
    dmrs.loc[(dmrs['meandiffabs']>=0.5)&(dmrs['DMTree'].fillna('').str.contains(i.replace('|','\|')+','))][['chr','start','stop_1','strand']].\
        to_csv(path+i.replace('|','_')+'.bed', sep='\t', header=False)
    dmrs.loc[(dmrs['meandiffabs']>=0.5)&(~dmrs['DMTree'].fillna('').str.contains(i.replace('|','\|')+','))][['chr','start','stop_1','strand']].\
        to_csv(path+i.replace('|','_')+'.anti.bed', sep='\t', header=False)
    os.system(env+homer_path+'findMotifsGenome.pl '+path+i.replace('|','_')+'.bed'+\
                  ' hg19'+\
                  ' '+path+i.replace('|','_')+' -bits -size 250  -bg '+path+i.replace('|','_')+'.anti.bed ')

In [ ]:
os.system('cp ./PDAC/motif/P2_3_3_3.bed ../SourceData/Fig.ED9a-part1.txt')
os.system('cp ./PDAC/motif/P2_3_3_3.anti.bed ../SourceData/Fig.ED9a-part2.txt')
os.system('cp ./PDAC/motif/N2_3_3_3.bed ../SourceData/Fig.ED9a-part3.txt')
os.system('cp ./PDAC/motif/N2_3_3_3.anti.bed ../SourceData/Fig.ED9a-part4.txt')

In [ ]:
for i in ['N2|3|3|3']:
    dmrs.loc[(dmrs['meandiffabs']>=0.5)&(dmrs['DMTree'].fillna('').str.contains(i.replace('|','\|')+','))][['chr','start','stop_1','strand']].\
        to_csv(path+i.replace('|','_')+'.bed', sep='\t', header=False)
    dmrs.loc[(dmrs['meandiffabs']>=0.5)&(~dmrs['DMTree'].fillna('').str.contains(i.replace('|','\|')+','))][['chr','start','stop_1','strand']].\
        to_csv(path+i.replace('|','_')+'.anti.bed', sep='\t', header=False)
    os.system(env+homer_path+'findMotifsGenome.pl '+path+i.replace('|','_')+'.bed'+\
                  ' hg19'+\
                  ' '+path+i.replace('|','_')+' -bits -size 250  -bg '+path+i.replace('|','_')+'.anti.bed ')

In [ ]:
os.system('cp ./PDAC/motif/P2_3_3_3/knownResults.html ./figures/ED9a.html')
os.system('cp ./PDAC/motif/N2_3_3_3/knownResults.html ./figures/ED9a-b.html')

In [ ]:
# PDAC-hyper DMRs
dmrs.loc[(dmrs['meandiffabs']>=0.5)&(dmrs['DMTree'].fillna('').str.contains("N2|3|3|3".replace('|','\|')+','))]['sig.comparison'].value_counts()

In [ ]:
# PDAC-hypo DMRs
dmrs.loc[(dmrs['meandiffabs']>=0.5)&(dmrs['DMTree'].fillna('').str.contains("P2|3|3|3".replace('|','\|')+','))]['sig.comparison'].value_counts()

In [ ]:
# PDAC-hypo DMRs
dmrs.loc[(dmrs['meandiffabs']>=0.5)&(~dmrs['DMTree'].fillna('').str.contains("P2|3|3|3".replace('|','\|')+','))]['sig.comparison'].value_counts().sum()

In [ ]:
motif_id = {}

os.system('mkdir ./PDAC/motif/find')

for motif in os.listdir('./PDAC/motif/P2_3_3_3/knownResults'):
    if motif.split('.')[-1]=='motif':
        motif_id[motif] = pd.read_table('./PDAC/motif/P2_3_3_3/knownResults/'+motif,nrows=0).columns[1]

        os.system(env+homer_path+'findMotifsGenome.pl ./PDAC/motif/P2_3_3_3.bed'+\
                  ' hg19'+\
                  ' ./PDAC/motif/find/'+motif+' \
                  -bits -size 250 \
                  -find  ./PDAC/motif/P2_3_3_3/knownResults/'+motif+' > \
                  ./PDAC/motif/find/'+motif+'.find.PDAC.tsv')
        
        os.system(env+homer_path+'findMotifsGenome.pl ./PDAC/motif/P2_3_3_3.anti.bed'+\
                  ' hg19'+\
                  ' ./PDAC/motif/find/'+motif+'-anti \
                  -bits -size 250 \
                  -find  ./PDAC/motif/P2_3_3_3/knownResults/'+motif+' > \
                  ./PDAC/motif/find/'+motif+'.find.PDAC.anti.tsv')
        
        print(motif,motif_id[motif])

In [ ]:
os.system('cp ./PDAC/motif/P2_3_3_3.bed ../SourceData/Fig.ED9a-part1.txt')
os.system('cp ./PDAC/motif/P2_3_3_3.anti.bed ../SourceData/Fig.ED9a-part2.txt')
os.system('cp ./PDAC/motif/N2_3_3_3.bed ../SourceData/Fig.ED9a-part3.txt')
os.system('cp ./PDAC/motif/N2_3_3_3.anti.bed ../SourceData/Fig.ED9a-part4.txt')

In [ ]:
pdac_motifs = pd.read_table('./PDAC/motif/P2_3_3_3.bed', header=None)

for i in motif_id.keys():
    tmp = pd.read_table('./PDAC/motif/find/'+i+'.find.PDAC.tsv', index_col='PositionID')
    pdac_motifs[motif_id[i]] = pdac_motifs[0].map(tmp['Offset'].to_dict())
pdac_motifs.to_csv('../SourceData/Fig.ED9bc-part1.txt',sep='\t')
pdac_motifs

In [ ]:
pdac_anti_motifs = pd.read_table('./PDAC/motif/P2_3_3_3.anti.bed', header=None)

for i in motif_id.keys():
    tmp = pd.read_table('./PDAC/motif/find/'+i+'.find.PDAC.anti.tsv', index_col='PositionID')
    pdac_anti_motifs[motif_id[i]] = pdac_anti_motifs[0].map(tmp['Offset'].to_dict())
pdac_anti_motifs.to_csv('../SourceData/Fig.ED9bc-part2.txt',sep='\t')
pdac_anti_motifs

In [ ]:
tmp = [list(pdac_motifs[3]-pdac_motifs[2])+\
      list(pdac_anti_motifs[3]-pdac_anti_motifs[2]),\
      ['PDAC-hypo' for i in pdac_motifs[3]]+['other' for i in pdac_anti_motifs[3]]]
tmp = pd.DataFrame(tmp).T
print(st.ranksums(tmp.loc[tmp[1]=='PDAC-hypo'][0],tmp.loc[tmp[1]!='PDAC-hypo'][0]))
plt.subplots(figsize=[2,4])
sns.boxplot(x=tmp[1],y=tmp[0],color='white', width=0.35)
plt.yscale('log')
plt.savefig('./figures/ED9d.pdf')

In [ ]:
tmp.columns = ['length','type']
tmp.to_csv('../SourceData/Fig.ED9d.txt', sep='\t')

In [ ]:
cls = sns.clustermap(pdac_motifs[motif_id.values()].isna().corr(),\
               vmin = -0.3, vmax = 0.3, center = 0, cmap='coolwarm', figsize=[20,20])
plt.savefig('./figures/ED9c.pdf', bbox_inches='tight')
sns.clustermap(pdac_anti_motifs[motif_id.values()].isna().corr(),\
               vmin = -0.3, vmax = 0.3, center = 0, cmap='coolwarm', figsize=[20,20],\
               col_linkage=cls.dendrogram_col.linkage, row_linkage=cls.dendrogram_row.linkage
              )
plt.savefig('./figures/ED9c-m.pdf', bbox_inches='tight')
sns.clustermap(pdac_motifs[motif_id.values()].isna().corr()-pdac_anti_motifs[motif_id.values()].isna().corr(),\
               vmin = -0.3, vmax = 0.3, center = 0, cmap='coolwarm', figsize=[20,20], method='ward',\
               col_linkage=cls.dendrogram_col.linkage, row_linkage=cls.dendrogram_row.linkage
              )
plt.savefig('./figures/ED9c-r.pdf', bbox_inches='tight')

In [ ]:
dmrs = pd.read_table('./PDAC/DMRs.tsv')
dmrs['strand'] = '+'
dmrs['stop'] = dmrs['stop']+1
dmrs.loc[dmrs['meandiffabs']>=0.5][['chr','start','stop','strand']].to_csv('./PDAC/motif/DMRs.high.bed', sep='\t', header=False)
dmrs.loc[dmrs['meandiffabs']>=0.5][['chr','start','stop','strand']]

In [ ]:
motifs_toshow = {
    'NFATC1':'./PDAC/motif/P2_3_3_3/knownResults/known6.motif',\
    'NFKB2':'./PDAC/motif/P2_3_3_3/knownResults/known3.motif'\
}

In [ ]:
for i in motifs_toshow.keys():  
    print(i, motif_id[motifs_toshow[i].split('/')[-1]])
    os.system(env+homer_path+'findMotifsGenome.pl ./PDAC/motif/DMRs.high.bed'+\
                  ' hg19'+\
                  ' ./PDAC/motif/'+i+' \
                  -bits -size 250 \
                  -find '+motifs_toshow[i]+' > \
                  ./PDAC/motif/'+i+'.find.allhighDMRs.tsv')

In [ ]:
os.system('cp ./PDAC/motif/P2_3_3_3/knownResults.html ./figures/ED9a-b.html')
os.system('cp ./PDAC/motif/N2_3_3_3/knownResults.html ./figures/ED9a.html')

In [ ]:
pdac_motifs = {}

for i in motifs_toshow.keys():
    tmp = pd.read_table('./PDAC/motif/'+i+'.find.allhighDMRs.tsv', index_col='PositionID')
    tmp2 = pd.read_table('./PDAC/motif/P2_3_3_3.bed', header=None)
    tmp2 = tmp2.loc[tmp2[0].isin(set(tmp.index.astype(int)))]

In [ ]:
for i in motifs_toshow.keys():  
    print(i, motif_id[motifs_toshow[i].split('/')[-1]])
    os.system(env+homer_path+'scanMotifGenomeWide.pl '+motifs_toshow[i]+' hg19 -bed > \
                      ./PDAC/motif/'+i+'.find.hg19.tsv')

In [ ]:
for i in motifs_toshow.keys():  
    print(i, motif_id[motifs_toshow[i].split('/')[-1]])
    os.system(env+homer_path+'findMotifsGenome.pl ./PDAC/motif/P2_3_3_3.bed'+\
                  ' hg19'+\
                  ' ./PDAC/motif/'+i+' \
                  -bits -size 250 \
                  -find '+motifs_toshow[i]+' > \
                  ./PDAC/motif/'+i+'.find.PDAC.tsv')

In [ ]:
for i in motifs_toshow.keys():
    tmp = pd.read_table('./PDAC/motif/'+i+'.find.PDAC.tsv', index_col='PositionID')
    tmp2 = pd.read_table('./PDAC/motif/P2_3_3_3.bed', header=None)
    tmp2 = tmp2.loc[tmp2[0].isin(set(tmp.index.astype(int)))]
    tmp2[5] = tmp2[0].map(tmp['Offset'].to_dict()).astype(int)
    tmp2[6] = ((tmp2[2]+tmp2[3])/2).astype(int)+tmp2[5]
    tmp2[7] = tmp2[6]+1
    tmp2[[1,6,7]].to_csv('./PDAC/motif/'+i+'.find.PDAC.emap.tsv',
                                           sep='\t', index=False,header=False)

In [ ]:
from pybedtools import BedTool
def find_overlapping_regions(bed1_df, bed2_df, bed1_cols, bed2_cols):
    bed_1 = BedTool.from_dataframe(bed1_df[bed1_cols].sort_values(bed1_cols))
    bed_2 = BedTool.from_dataframe(bed2_df[bed2_cols].sort_values(bed2_cols))
    return BedTool.to_dataframe(bed_1.intersect(bed_2, wa=True, wb=True))

def find_closest_regions(bed1_df, bed2_df, bed1_cols, bed2_cols):
    bed_1 = BedTool.from_dataframe(bed1_df[bed1_cols].sort_values(bed1_cols))
    bed_2 = BedTool.from_dataframe(bed2_df[bed2_cols].sort_values(bed2_cols))
    return BedTool.to_dataframe(bed_1.closest(bed_2, d=True))

In [ ]:
ratio_cob = {}
fl = 30

motifs_pos = {}
for ii,i in enumerate(['hg19','PDAC.emap']):
    motifs_pos[i] = {}
    for j in ['NFATC1','NFKB2']:
        motifs_pos[i][j] = pd.read_table('./PDAC/motif/'+j+'.find.'+i+'.tsv', header=None)
        motifs_pos[i][j][2] = motifs_pos[i][j][1]+1
        
for ii,i in enumerate(['hg19','PDAC.emap']):
    ratio_cob[i] = {}
    closest = find_closest_regions(motifs_pos[i]['NFATC1'],\
                                     motifs_pos[i]['NFKB2'],[0,1,2],[0,1,2]\
                                    )
    print(i,closest.shape[0],\
         closest.loc[(closest['thickStart']>=0)&(closest['thickStart']<(fl*2))].shape[0])

    ratio_cob[i]['NFATC1'] = closest.loc[(closest['thickStart']>=0)&(closest['thickStart']<(fl*2))].shape[0]/closest.shape[0]
    
    closest = find_closest_regions(motifs_pos[i]['NFKB2'],\
                                     motifs_pos[i]['NFATC1'],[0,1,2],[0,1,2]\
                                    )
    print(i,closest.shape[0],\
         closest.loc[(closest['thickStart']>=0)&(closest['thickStart']<(fl*2))].shape[0])

    ratio_cob[i]['NFKB2'] = closest.loc[(closest['thickStart']>=0)&(closest['thickStart']<(fl*2))].shape[0]/closest.shape[0]

pd.DataFrame(ratio_cob)

In [ ]:
allhighdmrs_motifs = {}

for i in motifs_toshow.keys():
    tmp = pd.read_table('./PDAC/motif/'+i+'.find.allhighDMRs.tsv', index_col='PositionID')
    allhighdmrs_motifs[i+'-dmrs'] = dmrs.loc[dmrs.index.isin(set(tmp.index.astype(int)))]
    allhighdmrs_motifs[i+'-dmrs']['DMRid'] = allhighdmrs_motifs[i+'-dmrs'].index
    allhighdmrs_motifs[i+'-dmrs'][5] = allhighdmrs_motifs[i+'-dmrs']['DMRid'].map(tmp['Offset'].to_dict()).astype(int)
    allhighdmrs_motifs[i+'-dmrs'][6] = ((allhighdmrs_motifs[i+'-dmrs']['start']+allhighdmrs_motifs[i+'-dmrs']['stop'])/2).astype(int)+allhighdmrs_motifs[i+'-dmrs'][5]
    allhighdmrs_motifs[i+'-dmrs'][7] = allhighdmrs_motifs[i+'-dmrs'][6]+1
    allhighdmrs_motifs[i+'-dmrs'][['chr',6,7]].to_csv('./PDAC/motif/'+i+'.find.allhighDMRs.emap.tsv',
                                           sep='\t', index=False,header=False)

In [ ]:
tmp = pd.merge(allhighdmrs_motifs['NFATC1-dmrs'],allhighdmrs_motifs['NFKB2-dmrs'],on=['chr','start','stop'])[['chr','6_x','7_x','6_y','7_y']]
tmp[6] = tmp[['6_x','7_x','6_y','7_y']].T.min()
tmp[7] = tmp[['6_x','7_x','6_y','7_y']].T.max()
tmp.to_csv('./PDAC/motif/NFKB2_NFATC1.find.allhighDMRs.emap.tsv',
                                           sep='\t', index=False,header=False)

In [ ]:
os.system('cp ./PDAC/motif/NFKB2_NFATC1.find.allhighDMRs.emap.tsv ../SourceData/Fig.7a.txt')
os.system('cp ./PDAC/motif/NFKB2.find.allhighDMRs.emap.tsv ../SourceData/Fig.ED9e-part1.txt')
os.system('cp ./PDAC/motif/NFATC1.find.allhighDMRs.emap.tsv ../SourceData/Fig.ED9e-part2.txt')

In [ ]:
highdmrs = dmrs.loc[(dmrs['meandiffabs']>=0.5)]
for i in ['NFATC1-dmrs','NFKB2-dmrs']:
    highdmrs[i] = (
        highdmrs['chr']+':'+highdmrs['start'].astype(str)
    ).isin(
        set(allhighdmrs_motifs[i]['chr']+':'+allhighdmrs_motifs[i]['start'].astype(str))
    )

print(st.fisher_exact(
    pd.crosstab(highdmrs['DMTree'].fillna('').str.contains('P2\|3\|3\|3'),\
    highdmrs['NFATC1-dmrs']&highdmrs['NFKB2-dmrs'])
))
pd.crosstab(highdmrs['DMTree'].fillna('').str.contains('P2\|3\|3\|3'),\
highdmrs['NFATC1-dmrs']&highdmrs['NFKB2-dmrs']).to_csv('./figures/ED9b-r.tsv',sep='\t')
pd.crosstab(highdmrs['DMTree'].fillna('').str.contains('P2\|3\|3\|3'),\
highdmrs['NFATC1-dmrs']&highdmrs['NFKB2-dmrs'])

In [ ]:
pd.crosstab(highdmrs['DMTree'].fillna('').str.contains('P2\|3\|3\|3'),\
highdmrs['NFATC1-dmrs']|highdmrs['NFKB2-dmrs'], margins=True)

In [ ]:
i = 'NFATC1-dmrs'
print(st.fisher_exact(
    pd.crosstab(highdmrs['DMTree'].fillna('').str.contains('P2\|3\|3\|3'),\
    highdmrs[i])
))
pd.crosstab(highdmrs['DMTree'].fillna('').str.contains('P2\|3\|3\|3'),\
highdmrs[i]).sort_index(ascending=False).T.sort_index(ascending=False).T.to_csv('./figures/ED9b-m.tsv',sep='\t')
pd.crosstab(highdmrs['DMTree'].fillna('').str.contains('P2\|3\|3\|3'),\
highdmrs[i]).sort_index(ascending=False).T.sort_index(ascending=False).T

In [ ]:
i = 'NFKB2-dmrs'
print(st.fisher_exact(
    pd.crosstab(highdmrs['DMTree'].fillna('').str.contains('P2\|3\|3\|3'),\
    highdmrs[i])
))
pd.crosstab(highdmrs['DMTree'].fillna('').str.contains('P2\|3\|3\|3'),\
highdmrs[i]).sort_index(ascending=False).T.sort_index(ascending=False).T.to_csv('./figures/ED9b.tsv',sep='\t')
pd.crosstab(highdmrs['DMTree'].fillna('').str.contains('P2\|3\|3\|3'),\
highdmrs[i]).sort_index(ascending=False).T.sort_index(ascending=False).T

In [ ]:
gepia2 = pd.read_table('./gepia2_paad_degs.txt', index_col=0, comment='#')
gepia2.columns = ['transcript','PAAD','Normal','Log2FC','P-value']
gepia2['DEcat'] = -1*(gepia2['Log2FC']<-1)+(gepia2['Log2FC']>1)
gepia2['SYMBOL'] = gepia2.index
gepia2

In [ ]:
venn2tab = highdmrs[['chr','start','stop','meandiffabs','SYMBOL','Hypo-groups','Int-groups','Hyper-groups','NFATC1-dmrs','NFKB2-dmrs']].sort_values(['chr','start'])
for i in ['Hypo-groups','Int-groups','Hyper-groups']:
    venn2tab[i] = venn2tab[i].str.replace('G0','PDAC')
    venn2tab[i] = venn2tab[i].str.replace('G1','PanIN')
    venn2tab[i] = venn2tab[i].str.replace('G2','Acinar')
    venn2tab[i] = venn2tab[i].str.replace('G3','Duct')
venn2tab.to_csv('./figures/ST8.tsv',sep='\t', index=False)
venn2tab

In [ ]:
volcano2tab = pd.merge(gepia2,venn2tab,on='SYMBOL',how='left')[['transcript','Log2FC','P-value','SYMBOL',\
                                       'chr','start','stop','meandiffabs','Hypo-groups','Int-groups','Hyper-groups','NFATC1-dmrs','NFKB2-dmrs']].sort_values('P-value')
volcano2tab.to_csv('./figures/ST9.tsv',sep='\t', index=False)
volcano2tab

In [ ]:
volcano2tab['P-value'] = -volcano2tab['P-value'].apply(np.log10)
volcano2tab = volcano2tab.loc[volcano2tab['NFATC1-dmrs']|volcano2tab['NFKB2-dmrs']].sort_values(['NFATC1-dmrs','NFKB2-dmrs'])
volcano2tab

In [ ]:
print(st.fisher_exact(pd.crosstab(volcano2tab['Log2FC']>0, volcano2tab['Hypo-groups'].str.contains('PDAC'))))
pd.crosstab(volcano2tab['Log2FC']>0, volcano2tab['Hypo-groups'].str.contains('PDAC'))

In [ ]:
plt.subplots(figsize=[3,3])
sns.scatterplot(x=gepia2['Log2FC'],y=-gepia2['P-value'].apply(np.log10), s=1,color='lightgrey')

volcano2tab['shape2'] = volcano2tab['NFATC1-dmrs']*1+volcano2tab['NFKB2-dmrs']*2

sns.scatterplot(x=volcano2tab['Log2FC'],\
                y=volcano2tab['P-value'], 
                s=10,color='grey',style=volcano2tab['shape2'],markers={1:'X',2:'P',3:'s'},legend=None)
sns.scatterplot(x=volcano2tab.loc[volcano2tab['Log2FC']>1]['Log2FC'],\
                y=volcano2tab.loc[volcano2tab['Log2FC']>1]['P-value'], 
                s=10,color='red',style=volcano2tab.loc[volcano2tab['Log2FC']>1]['shape2'],markers={1:'X',2:'P',3:'s'},legend=None)
sns.scatterplot(x=volcano2tab.loc[volcano2tab['Log2FC']<-1]['Log2FC'],\
                y=volcano2tab.loc[volcano2tab['Log2FC']<-1]['P-value'], 
                s=10,color='blue',style=volcano2tab.loc[volcano2tab['Log2FC']<-1]['shape2'],markers={1:'X',2:'P',3:'s'},legend=None)
plt.xlim([-12,12])
plt.yticks([0,50,100])
plt.savefig('./figures/7c.pdf', bbox_inches='tight')

In [ ]:
volcano2tab.to_csv('../SourceData/Fig.7c.txt',sep='\t')
volcano2tab

In [ ]:
met = pd.read_table('./data/GSE222147_pdac.input.tsv')
metcols = ['chr','pos']
cpgpos = met[['chr','pos']]
cpgpos['start'] = cpgpos['pos']
cpgpos['end'] = cpgpos['pos']+1
for i in ['PDAC','PanIN','Acinar','Duct']:
    met['pos0'] = met['pos']-1
    met['mean_'+i] = met[met.columns[met.columns.str.contains('='+i)]].T.mean()

met[['chr','pos','mean_PDAC','mean_PanIN','mean_Acinar','mean_Duct']].to_csv(\
    './PDAC/motif/groupmean.met.csv')

In [ ]:
met[['chr','pos','mean_PDAC','mean_PanIN','mean_Acinar','mean_Duct']].to_csv(\
    '../SourceData/Fig.6fg_ED10c.txt',sep='\t')

In [ ]:
os.system('Rscript ./enrichedheatmap.r')

In [ ]:
os.system('cp ./PDAC/motif/NFKB2_NFATC1.allhighDMRs.smooth.pdf ./figures/7a.pdf')
os.system('cp ./PDAC/motif/NFKB2.allhighDMRs.smooth.pdf ./figures/ED9e.pdf')
os.system('cp ./PDAC/motif/NFATC1.allhighDMRs.smooth.pdf ./figures/ED9e-b.pdf')

In [ ]:
import gseapy as gp

gs = sorted(set(venn2tab.loc[venn2tab['NFKB2-dmrs']]['SYMBOL'].dropna()))
print(len(gs), len(venn2tab.loc[venn2tab['NFKB2-dmrs']]['SYMBOL']))
gpres = gp.enrichr(gene_list=gs,
                 gene_sets='data/geo/h.all.v2023.2.Hs.symbols.gmt',
                 outdir=None,)
gpres = gpres.res2d.loc[gpres.res2d['P-value']<0.05]
plt.subplots(figsize=[3,3])
sns.barplot(y=gpres.sort_values('Odds Ratio', ascending=False)['Term'].str.replace('_',' ').str.replace('HALLMARK',''),\
            x=gpres.sort_values('Odds Ratio', ascending=False)['Odds Ratio'],\
           color='grey',width=0.6)
plt.savefig('./figures/ED9e-r.pdf')
gpres.to_csv('../SourceData/Fig.ED9e-part3.txt')
gpres

In [ ]:
import gseapy as gp

gs = sorted(set(venn2tab.loc[venn2tab['NFATC1-dmrs']]['SYMBOL'].dropna()))
print(len(gs), len(venn2tab.loc[venn2tab['NFATC1-dmrs']]['SYMBOL']))
gpres = gp.enrichr(gene_list=gs,
                 gene_sets='data/geo/h.all.v2023.2.Hs.symbols.gmt',
                 outdir=None,)
gpres = gpres.res2d.loc[gpres.res2d['P-value']<0.05]
plt.subplots(figsize=[3,3])
sns.barplot(y=gpres.sort_values('Odds Ratio', ascending=False)['Term'].str.replace('_',' ').str.replace('HALLMARK',''),\
            x=gpres.sort_values('Odds Ratio', ascending=False)['Odds Ratio'],\
           color='grey',width=0.6)
plt.savefig('./figures/ED9e-br.pdf')
gpres.to_csv('../SourceData/Fig.ED9e-part4.txt')
gpres

In [ ]:
def ndiff(a,b):
    if a and b:
        n = 0
        for i in range(len(a)):
            if a[i]!=b[i]:
                n += 1
        return n
    else:
        return None

dmrs['next_chr'] = list(dmrs['chr'][1:])+[None]
dmrs['next_start'] = list(dmrs['start'][1:])+[None]
dmrs['next_meandiffabs'] = list(dmrs['meandiffabs'][1:])+[None]
dmrs['nextnext_meandiffabs'] = list(dmrs['meandiffabs'][2:])+[None,None]
dmrs['next_sig.comparison'] = list(dmrs['sig.comparison'][1:])+[None]
dmrs['nextnext_sig.comparison'] = list(dmrs['sig.comparison'][2:])+[None,None]

dmrs['ndiff'] = dmrs.apply(lambda x:ndiff(x['sig.comparison'],x['next_sig.comparison']), axis=1)
dmrs['nndiff'] = dmrs.apply(lambda x:ndiff(x['next_sig.comparison'],x['nextnext_sig.comparison']), axis=1)

Llike = dmrs.loc[(dmrs['chr']==dmrs['next_chr'])&\
                ((dmrs['next_start']-dmrs['stop'])<1000)&\
                (dmrs['ndiff']==1)&\
                (dmrs['meandiffabs']>0.1)&(dmrs['next_meandiffabs']>0.1)]
Llike

In [ ]:
Llike_abba = Llike.loc[(Llike['sig.comparison'].apply(lambda x:(x[-3:] in ['3|1','1|3'])))&\
                        (Llike['next_sig.comparison'].apply(lambda x:(x[-3:] in ['3|1','1|3'])))&\
                        (Llike['next_sig.comparison'].apply(lambda x:x[-3:])==Llike['sig.comparison'].apply(lambda x:x[-3:]))&\
                        ((~Llike['sig.comparison'].str.contains('2'))&(~Llike['next_sig.comparison'].str.contains('2')))&\
                        (Llike['meandiffabs']>0.5)&\
                        (Llike['next_meandiffabs']>0.5)]
Llike_abba

In [ ]:
Llike_1133 = Llike.loc[(Llike['sig.comparison'].isin(['1|3|3|3','1|1|3|3']))&\
                        (Llike['next_sig.comparison'].isin(['1|3|3|3','1|1|3|3']))&\
                        (Llike['sig.comparison']==Llike['nextnext_sig.comparison'])&\
                        (Llike['meandiffabs']>0.5)&\
                        (Llike['next_meandiffabs']>0.5)&\
                        (Llike['nextnext_meandiffabs']>0.5)]

Llike_1133

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pybedtools import BedTool
def find_overlapping_regions_df(bed1_df, bed2_df, bed1_cols, bed2_cols):
    bed_1 = BedTool.from_dataframe(bed1_df[bed1_cols].sort_values(bed1_cols))
    bed_2 = BedTool.from_dataframe(bed2_df[bed2_cols].sort_values(bed2_cols))
    return BedTool.to_dataframe(bed_1.intersect(bed_2, wa=True, wb=True))

def plotIGV(tmp, cutoff=0):
    tmp['end'] = tmp['pos']+1
    dmrcpgs = find_overlapping_regions_df(tmp[['chr','pos','end']], dmrs.loc[dmrs['meandiffabs']>=cutoff][['chr','start','stop','meandiffabs','sig.comparison']], \
                                                    ['chr','pos','end'], ['chr','start','stop','meandiffabs','sig.comparison'])
          
    dmrcpgs.index = dmrcpgs['start']

    dmrs_tmp = dmrcpgs['name	score	strand'.split('\t')].drop_duplicates()
    
    for i in range(4):
        tmp[str(i)] = tmp['pos'].map(dmrcpgs['thickEnd'].apply(lambda x:int(x.split('|')[i])).to_dict()).fillna(0)
            
    f,d = plt.subplots(5,1,figsize=[10,5], sharex=True)
    
    for j,k in enumerate(['PDAC','PanIN','Acinar','Duct']):

        lenR = tmp['pos'].max()-tmp['pos'].min()
        sns.lineplot(x=[tmp['pos'].min()-0.01*lenR,tmp['pos'].max()+0.01*lenR],\
                                y=[1,1],color='#f0f0f0', linewidth=1,ax=d[j], zorder=0)
        sns.lineplot(x=[tmp['pos'].min()-0.01*lenR,tmp['pos'].max()+0.01*lenR],\
                                y=[0,0],color='#f0f0f0', linewidth=1,ax=d[j], zorder=0)
        
        sns.scatterplot(x=tmp['pos'],\
                        y=tmp[tmp.columns[tmp.columns.str.contains(k)]].T.mean(),color=typec[k],s=36, linewidth=0,ax=d[j])
        
        d[j].spines['top'].set_visible(False)
        d[j].spines['right'].set_visible(False)
        d[j].spines['bottom'].set_visible(False)
        d[j].xaxis.set_visible(False)
        d[j].set_ylim([-0.1,1.1])
        d[j].set_yticks([0,1])
        d[j].set_ylabel('')

        for dmri,dmr in dmrs_tmp.iterrows():
            d[j].axvspan(dmr['score'], dmr['strand'], alpha=0.1, color='red')
    
    # plt.subplots(figsize=[5,0.5])
    tmp.index = tmp['pos']
    
    tocolor = {'1':'#9dc2a9','2':'#e7e6e6','3':'#e4c198'}
    for j,i in dmrcpgs.drop_duplicates(['score','strand']).iterrows():
        for l,k in enumerate(i['thickEnd'].split('|')):
            sns.lineplot(x=[i['score'],i['strand']],\
                                y=[int(-l)/10+0.8,int(-l)/10+0.8],color=tocolor[k], linewidth=3.9,ax=d[-1])
    plt.ylim([0,1])
    d[-1].spines['top'].set_visible(False)
    d[-1].spines['right'].set_visible(False)
    d[-1].spines['left'].set_visible(False)
    d[-1].spines['bottom'].set_visible(False)
    d[-1].xaxis.set_visible(False)
    d[-1].yaxis.set_visible(False)
    
    print(ab[0]+':'+str(tmp['pos'].min())+'-'+str(tmp['pos'].max()))
    d[0].set_title(ab[0]+':'+str(tmp['pos'].min())+'-'+str(tmp['pos'].max())+'\n')
    
    d[0].set_xlim(tmp['pos'].min()-0.01*lenR,tmp['pos'].max()+0.01*lenR)

In [ ]:
ab = ['chr3',	42190587,	42190918]
tmp = met.loc[(met['chr']==ab[0])&(met['pos']>=(int(ab[1])-1e3))&(met['pos']<=(int(ab[-1])+3e3))]
plotIGV(tmp,0.)
plt.savefig('./figures/6f.pdf', bbox_inches='tight')

In [ ]:
ab = ['chr12', 104664289, 104666066]
tmp = met.loc[(met['chr']==ab[0])&(met['pos']>=(int(ab[1])-1e3))&(met['pos']<=(int(ab[-1])+3e3))]
plotIGV(tmp)
plt.savefig('./figures/6g.pdf', bbox_inches='tight')

In [ ]:
ab = ['chr11',	46400334,46407387]
tmp = met.loc[(met['chr']==ab[0])&(met['pos']>=(int(ab[1])-0))&(met['pos']<=(int(ab[-1])+0))]
plotIGV(tmp,0.5)
sns.lineplot(x=[46405509-10,46405509+10],y=[0.2,0.2],color='black',linewidth=2,estimator=False)
sns.lineplot(x=[46405548-10,46405548+10],y=[0.1,0.1],color='black',linewidth=2,estimator=False)
plt.savefig('./figures/ED10c.pdf', bbox_inches='tight')

In [ ]:
ganno = pd.read_table('./data/geo/Human.GRCh38.p13.annot.tsv.gz', index_col=0)
sanno = pd.read_table('./data/geo/GSE210351_series_matrix.txt.gz',\
                      skiprows=28, index_col=0).T
sanno['ID'] = sanno.index
sanno.index = sanno['!Sample_geo_accession']

liffers = pd.read_table('./data/geo/GSE210351_norm_counts_TPM_GRCh38.p13_NCBI.tsv.gz', index_col=0).T
liffers.index = liffers.index.map(sanno['ID'])
liffers.columns = liffers.columns.map(ganno['Symbol'])
liffers

In [ ]:
liffers['4type'] = [i.split('0')[0].split('-')[0][-1]+i.split('0')[0].split('-')[-1] for i in liffers.index]

In [ ]:
xena_meta = pd.read_table('data/geo/TCGA_GTEX_category.txt')
xena_meta.loc[xena_meta['TCGA_GTEX_main_category'].str.contains('Pancrea')]['TCGA_GTEX_main_category'].unique()

In [ ]:
tpmcols = pd.read_table('data/geo/TcgaTargetGtex_rsem_gene_tpm.gz', nrows=0, index_col=0).columns

pancrea = set(xena_meta.loc[xena_meta['TCGA_GTEX_main_category'].str.contains('Pancrea')]['sample'])

col2read = ['sample']
for i in tpmcols:
    if i in pancrea:
        col2read.append(i)

len(col2read)

In [ ]:
xena_tpm = pd.read_table('data/geo/TcgaTargetGtex_rsem_gene_tpm.gz', \
                         index_col=0, usecols=col2read)
xena_tpm

In [ ]:
xena_tpm = xena_tpm.T.applymap(lambda x:(2**x-0.001))
xena_tpm

In [ ]:
gepia2 = pd.read_table('./gepia2_paad_degs.txt', index_col=1, comment='#')
xena_tpm.columns = xena_tpm.columns.map(gepia2['Unnamed: 0'])
xena_tpm['4type'] = [i[:4] for i in xena_tpm.index]
xena_tpm

In [ ]:
twoSets = pd.merge(liffers.T,xena_tpm.T,left_index=True,right_index=True).T
twoSets

In [ ]:
twoSets.dropna()['4type'].value_counts()

In [ ]:
for ii,g in enumerate(['NFKB2','NFATC1']):
    import matplotlib.pyplot as plt
    typec = {
            'C':sns.color_palette("Paired")[5],
            'N':sns.color_palette("Paired")[8],
            'NH':sns.color_palette("Paired")[9],
            'G':sns.color_palette("Paired")[3],
            'GTEX':sns.color_palette("Paired")[3],
            'TCGA':sns.color_palette("Paired")[5],
        }
    
    f,ax = plt.subplots(1,2,figsize=[3,4],sharey=True, gridspec_kw={'width_ratios': [2, 1]})
    sns.boxplot(x=twoSets['4type'],y=twoSets[g].apply(lambda x:np.log2(x+1)),palette=typec,order=['G','N','NH','C',],color='white',ax=ax[0])
    sns.boxplot(x=twoSets['4type'],y=twoSets[g].apply(lambda x:np.log2(x+1)),palette=typec,order=['GTEX','TCGA'],color='white',ax=ax[1])
    ax[1].set_ylabel('')
    plt.savefig('./figures/7b-'+str(ii)+'.pdf')

In [ ]:
for ii,g in enumerate(['TRAK1','TXNRD1']):
    import matplotlib.pyplot as plt
    typec = {
            'C':sns.color_palette("Paired")[5],
            'N':sns.color_palette("Paired")[8],
            'NH':sns.color_palette("Paired")[9],
            'G':sns.color_palette("Paired")[3],
            'GTEX':sns.color_palette("Paired")[3],
            'TCGA':sns.color_palette("Paired")[5],
        }
    
    f,ax = plt.subplots(1,2,figsize=[3,4],sharey=True, gridspec_kw={'width_ratios': [2, 1]})
    sns.boxplot(x=twoSets['4type'],y=twoSets[g].apply(lambda x:np.log2(x+1)),palette=typec,order=['G','N','NH','C',],color='white',ax=ax[0])
    sns.boxplot(x=twoSets['4type'],y=twoSets[g].apply(lambda x:np.log2(x+1)),palette=typec,order=['GTEX','TCGA'],color='white',ax=ax[1])
    ax[1].set_ylabel('')
    plt.savefig('./figures/ED8a-'+str(ii)+'.pdf')

In [ ]:
for ii,g in enumerate(['NFKB1', 'RELA', 'RELB',  'NFATC2', 'NFATC3', 'NFATC4']):
    import matplotlib.pyplot as plt
    typec = {
            'C':sns.color_palette("Paired")[5],
            'N':sns.color_palette("Paired")[8],
            'NH':sns.color_palette("Paired")[9],
            'G':sns.color_palette("Paired")[3],
            'GTEX':sns.color_palette("Paired")[3],
            'TCGA':sns.color_palette("Paired")[5],
        }
    
    f,ax = plt.subplots(1,2,figsize=[3,4],sharey=True, gridspec_kw={'width_ratios': [2, 1]})
    sns.boxplot(x=twoSets['4type'],y=twoSets[g].apply(lambda x:np.log2(x+1)),palette=typec,order=['G','N','NH','C',],color='white',ax=ax[0])
    sns.boxplot(x=twoSets['4type'],y=twoSets[g].apply(lambda x:np.log2(x+1)),palette=typec,order=['GTEX','TCGA'],color='white',ax=ax[1])
    ax[1].set_ylabel('')
    plt.savefig('./figures/ED10a-'+str(ii)+'.pdf')

In [ ]:
g = 'MDK'
import matplotlib.pyplot as plt
typec = {
        'C':sns.color_palette("Paired")[5],
        'N':sns.color_palette("Paired")[8],
        'NH':sns.color_palette("Paired")[9],
        'G':sns.color_palette("Paired")[3],
        'GTEX':sns.color_palette("Paired")[3],
        'TCGA':sns.color_palette("Paired")[5],
    }

f,ax = plt.subplots(1,2,figsize=[3,4],sharey=True, gridspec_kw={'width_ratios': [2, 1]})
sns.boxplot(x=twoSets['4type'],y=twoSets[g].apply(lambda x:np.log2(x+1)),palette=typec,order=['G','N','NH','C',],color='white',ax=ax[0])
sns.boxplot(x=twoSets['4type'],y=twoSets[g].apply(lambda x:np.log2(x+1)),palette=typec,order=['GTEX','TCGA'],color='white',ax=ax[1])
ax[1].set_ylabel('')
plt.savefig('./figures/ED10b.pdf')

In [ ]:
twoSets[['NFKB2','NFATC1']+['TRAK1','TXNRD1']+['NFKB1', 'RELA', 'RELB',  'NFATC2', 'NFATC3', 'NFATC4']+['MDK']].to_csv('../SourceData/Fig.7b_ED8a_ED10ab.txt')
twoSets[['NFKB2','NFATC1']+['TRAK1','TXNRD1']+['NFKB1', 'RELA', 'RELB',  'NFATC2', 'NFATC3', 'NFATC4']+['MDK']]